# Clasificador semilla de discos lumbares (RM T2 sagital) · Entrenamiento y preetiquetado

**Proyecto:** Visión Artificial 2026-A · **Modelo:** YOLOv8-cls (Ultralytics) · **Clases:** Normal, Degeneración y Protrusión (= Hernia + Abombamiento fusionadas, ver `FUSIONES`); con `FUSIONES = {}` vuelve a las 4 originales

Este cuaderno entrena un clasificador **pequeño** con la semilla etiquetada a mano (Roboflow, versión v1) y lo usa para **preetiquetar** el dataset grande de ROIs, siempre con **revisión humana** de lo dudoso.

### Cómo usarlo
1. Entorno de ejecución → **GPU** (T4 sirve).
2. En Colab, panel de la izquierda → 🔑 **Secretos** → crea `ROBOFLOW_API_KEY` y activa *Acceso desde el cuaderno*. La clave nunca se escribe en el código.
3. Rellena workspace / proyecto / versión en la celda 2 (están en el snippet de Download Dataset de Roboflow, formato *Folder Structure* / Classification).
4. Ejecuta de arriba hacia abajo. Las secciones 1 a 6 entrenan y evalúan; la sección 7 preetiqueta.

### Hechos y límites (léelos antes de interpretar resultados)
**Hechos (de la versión v1 de Roboflow y la memoria del proyecto):**
- 780 imágenes: 720 train (aumentadas ×3: rotación ±5°, brillo ±15%) y 60 valid (originales, sin aumentar); sin test. Entrada 224×224 (estiramiento desde ROIs de 128×128).
- Valid tiene **15 imágenes por clase original** (30 en Protrusión si se fusiona). Las etiquetas son de estudiantes, sin radiólogo; sin corte axial.
- Un solo equipo de RM (Siemens 1.5 T). El detector `best_disc_detector.pt` solo ubica discos; **no se usa aquí**.

**Hipótesis (no verificadas):**
- Con 15 por clase, cualquier métrica tiene un margen de error amplio (ver intervalos de Wilson más abajo): diferencias de unos pocos puntos no son interpretables.
- **Fusión:** se decidió después de ver la matriz de 4 clases (post hoc), porque sin corte axial Hernia y Abombamiento se confundían (10 de 20 errores en la corrida de 4 clases). Entrena también con `FUSIONES = {}` y reporta ambos resultados.
- Casi toda la patología está en L5-S1, así que el modelo podría aprender atajos de nivel/posición en vez de patología.
- Un modelo entrenado con clases equilibradas **sobrepredice las clases raras** cuando se aplica a datos reales (donde Normal domina). Por eso el preetiquetado siempre pasa por revisión humana.
- El 95% de la rúbrica, medido aquí, sería acuerdo con las etiquetas de los estudiantes, no con la patología real.

## 0. Instalación y configuración

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
# ================== VARIABLES A RELLENAR ==================
RF_WORKSPACE = "TU_WORKSPACE"   # <- slug del workspace en Roboflow
RF_PROJECT   = "TU_PROYECTO"    # <- slug del proyecto de clasificación
RF_VERSION   = 1                # versión v1
SECRET_NAME  = "ROBOFLOW_API_KEY"  # nombre del secreto en Colab (no la clave)

# Fusión de clases (decisión del equipo: sin corte axial, Hernia y Abombamiento no se separan bien en sagital)
# {} = 4 clases originales (línea base para el informe). Los nombres deben coincidir con las carpetas de Roboflow.
FUSIONES = {"Abombamiento": "Protrusion", "Hernia_Discal": "Protrusion"}

# Entrenamiento
MODELO_BASE = "yolov8s-cls.pt"  # alternativas: yolov8n-cls.pt (más ligero), yolov8m-cls.pt
EPOCAS      = 60
PACIENCIA   = 15
BATCH       = 32
IMGSZ       = 224
SEMILLA     = 42

# Salidas en Google Drive
DRIVE_DIR = "/content/drive/MyDrive/VisionArtificial/preetiquetado"

# Preetiquetado
UMBRAL_CONF   = 0.70   # confianza top-1 por debajo de esto -> revisión humana
UMBRAL_MARGEN = 0.20   # (top1 - top2) por debajo de esto -> revisión humana
# ==========================================================
assert "TU_" not in RF_WORKSPACE + RF_PROJECT, "Rellena RF_WORKSPACE y RF_PROJECT."

In [ ]:
import os, re, json, random, shutil, datetime, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, ultralytics

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
fijar_semilla(SEMILLA)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (activa una en Entorno de ejecución)")
print("ultralytics", ultralytics.__version__, "· torch", torch.__version__)

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

## 1. Descarga del dataset desde Roboflow
La clave se lee de *Colab Secrets*; no queda en el cuaderno.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

try:
    api_key = userdata.get(SECRET_NAME)
except Exception as e:
    raise RuntimeError(f"No encuentro el secreto '{SECRET_NAME}'. Créalo en 🔑 Secretos y activa el acceso desde el cuaderno.") from e

rf = Rf = Roboflow(api_key=api_key)
version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
dataset = version.download("folder", location="/content/dataset_rf")  # formato Classification (carpetas por clase)
DATA = Path(dataset.location)
print("Dataset en:", DATA, "· carpetas:", sorted(p.name for p in DATA.iterdir() if p.is_dir()))

In [ ]:
# Ultralytics solo reconoce 'val' / 'validation' / 'test'; Roboflow exporta 'valid'.
if not (DATA / "val").exists() and (DATA / "valid").exists():
    os.symlink(DATA / "valid", DATA / "val")
assert (DATA / "train").exists() and (DATA / "val").exists(), "Faltan carpetas train/valid."

### Fusión de clases
Si `FUSIONES` no está vacío, se crea una copia del dataset con las clases fusionadas. Cada archivo conserva la **clase fina original como prefijo** (`Hernia_Discal__...`), para poder desglosar después qué había dentro de "Protrusion". La etiqueta fina no se pierde.

In [ ]:
DATA_RF = DATA
CLASES_RF = sorted(p.name for p in (DATA_RF / "train").iterdir() if p.is_dir())
faltan = set(FUSIONES) - set(CLASES_RF)
assert not faltan, f"FUSIONES menciona clases que no existen en el dataset: {faltan}. Clases reales: {CLASES_RF}"

if FUSIONES:
    DATA = Path("/content/dataset_fusionado")
    shutil.rmtree(DATA, ignore_errors=True)
    for split, src in (("train", "train"), ("val", "valid")):
        for cdir in (DATA_RF / src).iterdir():
            if not cdir.is_dir(): continue
            dest = DATA / split / FUSIONES.get(cdir.name, cdir.name)
            dest.mkdir(parents=True, exist_ok=True)
            for f in cdir.iterdir():
                shutil.copy(f, dest / f"{cdir.name}__{f.name}")
    print("Dataset fusionado en:", DATA, "· clases:", sorted(p.name for p in (DATA / "train").iterdir()))
else:
    print("Sin fusión: se usan las clases originales:", CLASES_RF)

## 2. Verificación del dataset
Cuenta por clase y comprobaciones de fuga. Si el nombre del archivo conserva el ID de paciente (`__0339_008`), se comprueba que ningún paciente esté a la vez en train y valid. Si Roboflow renombró los archivos, aparecerá un aviso y la comprobación **no** aplica.

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp"}
RE_ID = re.compile(r"__(\d{4})_(\d{3})_")
RE_NIVEL = re.compile(r"(L[1-5]-(?:L[1-5]|S1))")

def listar(split):
    filas = []
    for cdir in sorted((DATA / split).iterdir()):
        if not cdir.is_dir(): continue
        for f in cdir.iterdir():
            if f.suffix.lower() in EXT:
                m, n = RE_ID.search(f.name), RE_NIVEL.search(f.name)
                filas.append(dict(split=split, clase=cdir.name, archivo=f.name, ruta=str(f),
                                  clase_fina=f.name.partition("__")[0] if FUSIONES else cdir.name,
                                  paciente=m.group(1) if m else None,
                                  nivel=n.group(1) if n else None))
    return pd.DataFrame(filas)

df_tr, df_va = listar("train"), listar("val")
df_all = pd.concat([df_tr, df_va], ignore_index=True)
print(pd.crosstab(df_all["clase"], df_all["split"], margins=True))

print("\nClases finas (antes de fusionar):"); display(pd.crosstab(df_all["clase_fina"], df_all["split"]))
n_va = df_va.clase.value_counts()
print("\nValid por clase:", n_va.to_dict())
print("⚠ Valid tiene solo 15 imágenes por clase fina: cada recall/precisión se apoya en pocos casos. Margen de error amplio.")
if FUSIONES:
    print("⚠ Tras fusionar, 'Protrusion' pesa el doble que las otras clases en train y valid: lee F1 macro y recall por clase, no solo la exactitud.")

if df_tr.paciente.notna().all() and df_va.paciente.notna().all():
    comun = set(df_tr.paciente) & set(df_va.paciente)
    print(f"Pacientes en train: {df_tr.paciente.nunique()} · en valid: {df_va.paciente.nunique()} · en ambos: {len(comun)}")
    assert not comun, f"FUGA: {len(comun)} pacientes aparecen en train y valid: {sorted(comun)[:10]}"
else:
    print("\n⚠ Los nombres no conservan el ID de paciente: no puedo verificar fuga train/valid. Hazlo con el script del split por paciente.")

# Chequeo visual rápido
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, (cl, g) in zip(axs.ravel()[:4], df_tr.groupby("clase")):
    ax.imshow(plt.imread(g.ruta.iloc[0]), cmap="gray"); ax.set_title(f"train · {cl}"); ax.axis("off")
for ax, (cl, g) in zip(axs.ravel()[4:], df_va.groupby("clase")):
    ax.imshow(plt.imread(g.ruta.iloc[0]), cmap="gray"); ax.set_title(f"valid · {cl}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Entrenamiento (YOLOv8-cls, fine-tuning)
- **Sin flips** (`fliplr=0`, `flipud=0`): en sagital, anterior/posterior tiene significado anatómico; invertirlo crea imágenes imposibles.
- El train ya viene aumentado ×3 desde Roboflow, así que el aumento en línea se deja mínimo (sin RandAugment, sin borrado aleatorio).
- Semilla fija (`seed` + `deterministic=True`). Aun así, GPU distinta o versión distinta de librerías puede dar diferencias pequeñas: **hipótesis** de que es reproducible bit a bit; lo seguro es que sea reproducible en resultado aproximado.
- Hechos a considerar: las 3 copias aumentadas de un mismo ROI están todas en train, por lo que la pérdida de entrenamiento es más optimista que la de valid. Decide con valid, no con train.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODELO_BASE)
RUN_NAME = "semilla_v%s_%s" % (RF_VERSION, datetime.datetime.now().strftime("%Y%m%d_%H%M"))

resultados = model.train(
    data=str(DATA), imgsz=IMGSZ, epochs=EPOCAS, patience=PACIENCIA, batch=BATCH,
    seed=SEMILLA, deterministic=True, workers=2,
    project="/content/runs", name=RUN_NAME, exist_ok=True,
    # Aumento en línea: mínimo y sin flips
    fliplr=0.0, flipud=0.0, auto_augment=None, erasing=0.0,
    degrees=0.0, translate=0.0, scale=0.0, shear=0.0,
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.05,
    plots=True,
)
RUN_DIR = Path("/content/runs") / RUN_NAME
print("Run en:", RUN_DIR)

In [ ]:
res = pd.read_csv(RUN_DIR / "results.csv"); res.columns = [c.strip() for c in res.columns]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].plot(res["epoch"], res["train/loss"], label="train"); axs[0].plot(res["epoch"], res["val/loss"], label="valid")
axs[0].set_title("Pérdida"); axs[0].legend()
axs[1].plot(res["epoch"], res["metrics/accuracy_top1"]); axs[1].set_title("Exactitud top-1 en valid (60 imágenes)")
plt.tight_layout(); plt.show()

## 4. Evaluación en valid: por clase, no solo exactitud global
Se usa el mejor punto (`best.pt`, elegido por exactitud en valid). **Aviso:** como valid también selecciona el mejor epoch y no hay test, las métricas están algo infladas (selección sobre el mismo conjunto). Dilo así en el informe.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from statsmodels.stats.proportion import proportion_confint

best = YOLO(RUN_DIR / "weights" / "best.pt")
NOMBRES = [best.names[i] for i in sorted(best.names)]

def predecir(modelo, rutas, batch=64):
    # devuelve matriz de probabilidades (n, n_clases) en el orden de modelo.names
    P = []
    for i in range(0, len(rutas), batch):
        for r in modelo.predict(rutas[i:i+batch], imgsz=IMGSZ, verbose=False):
            P.append(r.probs.data.cpu().numpy())
    return np.vstack(P)

P_va = predecir(best, df_va.ruta.tolist())
y_true = df_va.clase.map({n: i for i, n in enumerate(NOMBRES)}).values
y_pred = P_va.argmax(1)
assert not np.isnan(y_true.astype(float)).any(), "Nombres de clase de valid no coinciden con los del modelo."

print(classification_report(y_true, y_pred, target_names=NOMBRES, digits=3, zero_division=0))
exact = (y_true == y_pred).mean()
lo, hi = proportion_confint((y_true == y_pred).sum(), len(y_true), method="wilson")
print(f"Exactitud global: {exact:.3f}  (IC95% Wilson {lo:.2f}–{hi:.2f}, n={len(y_true)})")

In [ ]:
# Recall por clase con IC95% de Wilson: ilustra lo ancho del margen con n=15
filas = []
for i, n in enumerate(NOMBRES):
    k, tot = int(((y_pred == i) & (y_true == i)).sum()), int((y_true == i).sum())
    l, h = proportion_confint(k, tot, method="wilson") if tot else (np.nan, np.nan)
    filas.append(dict(clase=n, n_valid=tot, aciertos=k, recall=k/tot if tot else np.nan, IC95_inf=l, IC95_sup=h))
tabla_recall = pd.DataFrame(filas).round(3); display(tabla_recall)

cm = confusion_matrix(y_true, y_pred, labels=range(len(NOMBRES)))
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(NOMBRES)), NOMBRES, rotation=30, ha="right"); ax.set_yticks(range(len(NOMBRES)), NOMBRES)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xlabel("Predicha"); ax.set_ylabel("Real (etiqueta de estudiantes)"); ax.set_title("Matriz de confusión · valid")
plt.tight_layout(); plt.savefig(RUN_DIR / "matriz_confusion_valid.png", dpi=150); plt.show()

print("\n⚠ Lectura prudente: con 15 casos por clase fina, un caso mueve el recall varios puntos.")
if FUSIONES:
    print("Desglose de lo fusionado (clase fina real vs predicha):")
    display(pd.crosstab(df_va.clase_fina, [NOMBRES[i] for i in y_pred]))
    print("Si Hernia y Abombamiento se predicen bien como 'Protrusion', la fusión cumple su objetivo.")
else:
    print("Sin axial, Abombamiento vs Hernia no se separa bien; errores entre ellas son esperables.")

In [ ]:
# Por nivel (riesgo de atajo: la patología se concentra en L5-S1). Solo si el nombre trae el nivel.
if df_va.nivel.notna().all():
    d = df_va.assign(ok=(y_true == y_pred))
    display(d.groupby("nivel").agg(n=("ok", "size"), exactitud=("ok", "mean")).round(2))
    print("Hipótesis: si la exactitud cae fuera de L5-S1 o la hernia se predice solo en L5-S1, puede haber atajo de posición.")
else:
    print("Nombres sin nivel: no se puede desglosar por nivel.")

errores = df_va.assign(pred=[NOMBRES[i] for i in y_pred], conf=P_va.max(1).round(3))
errores = errores[errores.clase != errores.pred][["archivo", "clase", "pred", "conf"]]
print(f"\n{len(errores)} errores en valid (revisa estas imágenes a ojo):"); display(errores)

## 5. Calibración de los umbrales de revisión
La confianza softmax suele estar sobreestimada, y más con un dataset tan pequeño. Esta tabla muestra, **sobre valid**, cuánto se mandaría a revisión con los umbrales actuales y cómo acierta el modelo en lo que *no* se revisa. Es una guía, no una garantía (n=60).

In [ ]:
def marcar(P, umbral_conf=UMBRAL_CONF, umbral_margen=UMBRAL_MARGEN):
    top2 = np.sort(P, axis=1)[:, ::-1][:, :2]
    return (top2[:, 0] < umbral_conf) | ((top2[:, 0] - top2[:, 1]) < umbral_margen)

rev = marcar(P_va)
ok = (y_true == y_pred)
print(f"Umbrales: conf<{UMBRAL_CONF} o margen<{UMBRAL_MARGEN}")
print(f"A revisión: {rev.sum()}/{len(rev)} ({rev.mean():.0%})")
if (~rev).any(): print(f"Exactitud en lo NO marcado: {ok[~rev].mean():.2f} (n={(~rev).sum()})")
if rev.any():    print(f"Exactitud en lo marcado:    {ok[rev].mean():.2f} (n={rev.sum()})")

filas = []
for uc in [0.5, 0.6, 0.7, 0.8, 0.9]:
    r = marcar(P_va, uc, UMBRAL_MARGEN)
    filas.append(dict(umbral_conf=uc, pct_a_revision=round(r.mean(), 2),
                      exactitud_no_marcado=round(ok[~r].mean(), 2) if (~r).any() else np.nan))
display(pd.DataFrame(filas))

## 6. Guardar en Google Drive
Pesos, curvas, matriz de confusión y un JSON con la configuración exacta (para el informe y la reproducibilidad).

In [ ]:
DEST = Path(DRIVE_DIR) / RUN_NAME
(DEST / "weights").mkdir(parents=True, exist_ok=True)
shutil.copy(RUN_DIR / "weights" / "best.pt", DEST / "weights" / "best.pt")
shutil.copy(RUN_DIR / "weights" / "last.pt", DEST / "weights" / "last.pt")
for f in ["results.csv", "results.png", "matriz_confusion_valid.png", "args.yaml"]:
    if (RUN_DIR / f).exists(): shutil.copy(RUN_DIR / f, DEST / f)
tabla_recall.to_csv(DEST / "recall_por_clase_valid.csv", index=False)
errores.to_csv(DEST / "errores_valid.csv", index=False)

meta = dict(
    fecha=datetime.datetime.now().isoformat(timespec="seconds"),
    roboflow=dict(workspace=RF_WORKSPACE, proyecto=RF_PROJECT, version=RF_VERSION),
    modelo_base=MODELO_BASE, epocas=EPOCAS, paciencia=PACIENCIA, batch=BATCH, imgsz=IMGSZ, semilla=SEMILLA,
    clases=NOMBRES, fusiones=FUSIONES, n_train=len(df_tr), n_valid=len(df_va),
    exactitud_valid=round(float(exact), 4), exactitud_IC95_wilson=[round(lo, 3), round(hi, 3)],
    f1_por_clase={n: round(float(f), 3) for n, f in zip(NOMBRES, precision_recall_fscore_support(y_true, y_pred, labels=range(len(NOMBRES)), zero_division=0)[2])},
    umbrales_revision=dict(conf=UMBRAL_CONF, margen=UMBRAL_MARGEN),
    versiones=dict(python=platform.python_version(), torch=torch.__version__, ultralytics=ultralytics.__version__),
    advertencias=["valid=15 por clase fina (Protrusion=30 si se fusiona)", "fusión de clases decidida después de ver la matriz de 4 clases (post hoc)", "sin test; best.pt elegido sobre valid", "etiquetas de estudiantes sin radiólogo",
                  "entrenado con clases equilibradas: sobrepredice clases raras en datos reales"],
)
(DEST / "metadata.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado en Drive:", DEST)

## 7. Preetiquetado de una carpeta de ROIs
Sube (o monta desde Drive) una carpeta con ROIs `.png/.jpg` y el cuaderno escribe un CSV con clase predicha, confianza, segunda opción y una marca de **revisión humana**.

**Reglas del preetiquetado (importantes):**
- La predicción es una **sugerencia**, no una etiqueta. Un humano confirma o corrige; para no anclarse, audita ~10% de lo *no marcado* sin mirar la sugerencia.
- Se manda a revisión lo de confianza baja, margen top-1/top-2 pequeño, y cualquier predicción de **Protrusión** (o Hernia/Abombamiento sin fusión) si `REVISAR_CLASES_RARAS=True`.
- ⚠ **Sobrepredicción de clases raras:** el modelo se entrenó con clases equilibradas (Protrusión pesa el doble si se fusiona), pero en datos reales Normal domina (en los 70 de valid etiquetados: Normal 40, Hernia 17, Abombamiento 10, Degeneración 3, y es una muestra con sesgo). Espera **más falsos positivos de Hernia/Abombamiento/Degeneración** de los que sugiere la precisión en valid.
- Opcional: `PRIORS_REALES` ajusta las probabilidades a una prevalencia supuesta (corrección de prior). **Hipótesis:** solo es válida si las probabilidades están calibradas, cosa que no se ha verificado; úsala solo para comparar, nunca sin revisión.

In [ ]:
# ============ PARÁMETROS DE PREETIQUETADO ============
CARPETA_ROIS = "/content/drive/MyDrive/VisionArtificial/rois_a_preetiquetar"  # <- carpeta con los ROIs
PESOS        = str(DEST / "weights" / "best.pt")   # o la ruta a unos pesos guardados antes en Drive
CSV_SALIDA   = str(Path(DRIVE_DIR) / f"preetiquetas_{datetime.datetime.now():%Y%m%d_%H%M}.csv")
REVISAR_CLASES_RARAS = True   # enviar siempre a revisión lo predicho como Protrusion (o Hernia/Abombamiento sin fusión)
PRIORS_REALES = None          # p. ej. {"Normal":0.6,"Degeneración":0.2,"Abombamiento":0.1,"Hernia":0.1} (hipótesis); None = sin ajuste
# =====================================================

In [ ]:
modelo = YOLO(PESOS)
nombres = [modelo.names[i] for i in sorted(modelo.names)]
rutas = sorted(str(p) for p in Path(CARPETA_ROIS).rglob("*") if p.suffix.lower() in EXT)
assert rutas, f"No hay imágenes en {CARPETA_ROIS}"
print(f"{len(rutas)} ROIs a preetiquetar")

P = predecir(modelo, rutas)
P_orig = P.copy()
if PRIORS_REALES:
    w = np.array([PRIORS_REALES[n] / (1 / len(nombres)) for n in nombres])  # prior real / prior de entrenamiento (uniforme)
    P = P * w; P = P / P.sum(1, keepdims=True)

orden = np.argsort(-P, axis=1)
i1, i2 = orden[:, 0], orden[:, 1]
c1, c2 = P[np.arange(len(P)), i1], P[np.arange(len(P)), i2]

out = pd.DataFrame(dict(
    archivo=[Path(r).name for r in rutas],
    nivel=[(RE_NIVEL.search(Path(r).name).group(1) if RE_NIVEL.search(Path(r).name) else "") for r in rutas],
    clase_predicha=[nombres[i] for i in i1], confianza=c1.round(4),
    top2_clase=[nombres[i] for i in i2], top2_confianza=c2.round(4), margen=(c1 - c2).round(4),
))
motivos = []
for k in range(len(out)):
    m = []
    if c1[k] < UMBRAL_CONF: m.append(f"conf<{UMBRAL_CONF}")
    if c1[k] - c2[k] < UMBRAL_MARGEN: m.append(f"margen<{UMBRAL_MARGEN}")
    if REVISAR_CLASES_RARAS and out.clase_predicha[k].startswith(("Hernia", "Abombamiento", "Protrusion")): m.append("clase_rara")
    motivos.append("+".join(m))
out["revisar_humano"] = [bool(m) for m in motivos]
out["motivo_revision"] = motivos
# columnas vacías para que el revisor complete (no ver la sugerencia en la auditoría ciega)
out["clase_final_humana"] = ""
out["comentario"] = ""

out.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
print("CSV:", CSV_SALIDA)
print(f"A revisión humana: {out.revisar_humano.sum()}/{len(out)} ({out.revisar_humano.mean():.0%})")
display(pd.crosstab(out.clase_predicha, out.revisar_humano, margins=True))
display(out.head(10))

In [ ]:
# Auditoría ciega: ~10% de lo NO marcado, para estimar el error oculto del preetiquetado
no_marcado = out[~out.revisar_humano]
n_aud = max(1, int(round(0.10 * len(no_marcado)))) if len(no_marcado) else 0
if n_aud:
    aud = no_marcado.sample(n_aud, random_state=SEMILLA)[["archivo", "nivel", "clase_final_humana", "comentario"]]
    ruta_aud = str(Path(CSV_SALIDA).with_name(Path(CSV_SALIDA).stem + "_auditoria_ciega.csv"))
    aud.to_csv(ruta_aud, index=False, encoding="utf-8-sig")
    print(f"{n_aud} ROIs para auditar SIN ver la sugerencia (sin columna de predicción) →", ruta_aud)

## Siguientes pasos sugeridos
1. Revisar los errores de valid (sección 4) a ojo: ¿son etiquetas dudosas o fallos del modelo?
2. Corregir lo marcado, auditar el 10% ciego, y calcular el error del preetiquetado sobre lo auditado.
3. Añadir lo corregido a train (por paciente, sin tocar valid), reentrenar y repetir (active learning).
4. En el informe: reportar F1/recall por clase con sus intervalos, declarar valid=15/clase, ausencia de test, etiquetas sin radiólogo, un solo equipo y la sobrepredicción de clases raras.